## Imports and env. variables

In [1]:
# import dependencies
import os.path

# set path here
os.chdir("path to DeePEn")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from torch.utils.data import DataLoader

import re
import numpy as np
import pandas as pd
import copy

import transformers, datasets
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.models.t5.modeling_t5 import T5Config, T5PreTrainedModel, T5Stack
from transformers.utils.model_parallel_utils import assert_device_map, get_device_map
from transformers import T5EncoderModel, T5Tokenizer
from transformers import TrainingArguments, Trainer, set_seed
from transformers import EarlyStoppingCallback

from typing import Optional, Tuple, Union

from transformers.trainer_callback import TrainerCallback

import peft
from peft import get_peft_config, PeftModel, PeftConfig, inject_adapter_in_model, LoraConfig, PeftModelForSequenceClassification

from evaluate import load
from datasets import Dataset

from tqdm import tqdm
import random

from scipy import stats
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import json
import itertools
import time
import subprocess

# Environment to run this notebook


These are the versions of the core packages we use to run this notebook:

In [2]:
print("Torch version: ",torch.__version__)
print("Cuda version: ",torch.version.cuda)
print("Numpy version: ",np.__version__)
print("Pandas version: ",pd.__version__)
print("Transformers version: ",transformers.__version__)
print("Datasets version: ",datasets.__version__)

Torch version:  2.7.0+cu126
Cuda version:  12.6
Numpy version:  2.0.2
Pandas version:  2.2.3
Transformers version:  4.51.3
Datasets version:  3.5.1


### Select your model:

In [3]:
checkpoint = "Rostlab/prot_t5_xl_uniref50"

# Input data

Provide your training and validation data in seperate pandas dataframes 

In [4]:
# run prepare_data.py if the splits were not created yet
if not os.path.isdir('./data/splits'):
    subprocess.run(['python', './data/prepare_data.py'])

def read_data(path):
    
    df_train = pd.read_csv("./data/splits/" + path + "/train.csv")
    df_valid = pd.read_csv("./data/splits/" + path + "/valid.csv")
    
    df_train = df_train.rename(columns={"mutated_sequence" : "sequence", "DMS_score" : "label"})
    df_valid = df_valid.rename(columns={"mutated_sequence" : "sequence", "DMS_score" : "label"})
    
    # Initialize the scaler
    scaler = StandardScaler()

    # Fit the scaler on the training labels and transform the training labels
    df_train['label'] = scaler.fit_transform(df_train[['label']])

    # Use the same scaler to transform the validation labels
    df_valid['label'] = scaler.transform(df_valid[['label']])
    
    return df_train, df_valid

# Models and Low Rank Adaptation

## T5 Models

### Binary Ranking model definition 

adding a ranking head (num_labels = 1) on top of the encoder model

Using binary cross entropy (BCE) loss on the model logit outputs combined with probabilistic soft-labels 

Adding a onesided contrastive loss term (based on cosine similarity between embeddings) to only punish the case where very similar embeddings are paired with dissimilar fitness


In [5]:
class ClassConfig:
    def __init__(self, dropout=0.2, num_labels=1):
        self.dropout_rate = dropout
        self.num_labels = num_labels

class T5EncoderClassificationHead(nn.Module):
    """Head for sentence-level classification tasks."""

    def __init__(self, config, class_config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(class_config.dropout_rate)
        self.out_proj = nn.Linear(config.hidden_size, class_config.num_labels)

    def forward(self, hidden_states):

        hidden_states =  torch.mean(hidden_states,dim=1)  # avg embedding

        hidden_states = self.dropout(hidden_states)
        hidden_states = self.dense(hidden_states)
        hidden_states = torch.tanh(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.out_proj(hidden_states)
        return hidden_states

class T5EncoderForSimpleSequenceToRank(T5PreTrainedModel):

    def __init__(self, config: T5Config, class_config):
        super().__init__(config)
        self.num_labels = class_config.num_labels
        self.config = config

        self.shared = nn.Embedding(config.vocab_size, config.d_model)

        encoder_config = copy.deepcopy(config)
        encoder_config.use_cache = False
        encoder_config.is_encoder_decoder = False
        self.encoder = T5Stack(encoder_config, self.shared)

        self.dropout = nn.Dropout(class_config.dropout_rate)
        self.classifier = T5EncoderClassificationHead(config, class_config)

        # Initialize weights and apply final processing
        self.post_init()

        # Model parallel
        self.model_parallel = False
        self.device_map = None

    def parallelize(self, device_map=None):
        self.device_map = (
            get_device_map(len(self.encoder.block), range(torch.cuda.device_count()))
            if device_map is None
            else device_map
        )
        assert_device_map(self.device_map, len(self.encoder.block))
        self.encoder.parallelize(self.device_map)
        self.classifier = self.classifier.to(self.encoder.first_device)
        self.model_parallel = True

    def deparallelize(self):
        self.encoder.deparallelize()
        self.encoder = self.encoder.to("cpu")
        self.model_parallel = False
        self.device_map = None
        torch.cuda.empty_cache()

    def get_input_embeddings(self):
        return self.shared

    def set_input_embeddings(self, new_embeddings):
        self.shared = new_embeddings
        self.encoder.set_input_embeddings(new_embeddings)

    def get_encoder(self):
        return self.encoder

    def _prune_heads(self, heads_to_prune):
        """
        Prunes heads of the model. heads_to_prune: dict of {layer_num: list of heads to prune in this layer} See base
        class PreTrainedModel
        """
        for layer, heads in heads_to_prune.items():
            self.encoder.layer[layer].attention.prune_heads(heads)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if input_ids.shape[0] == 1:

            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                inputs_embeds=inputs_embeds,
                head_mask=head_mask,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )

            hidden_states = outputs[0]

            hidden_states = self.dropout(hidden_states)

            logits = self.classifier(hidden_states)

            loss = None
            if labels is not None:

                print("Can not do ranking training with a single example")

                loss = None

        # ranking loss between all parts of the "minibatch"
        else:
          # Compute outputs for the entire batch at once
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                inputs_embeds=inputs_embeds,
                head_mask=head_mask,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
                )

            hidden_states = outputs[0]

            hidden_states = self.dropout(hidden_states)

            logits = self.classifier(hidden_states)

            loss = None

            if labels is not None:
                
                def calculate_probability(label_i, label_j, sigma):
                    """
                    Calculate the probability that label_i is greater than label_j
                    assuming Gaussian noise with standard deviation sigma.
                    """
                    mu_diff = label_i - label_j
                    sigma_diff = sigma * (2 ** 0.5)  # sqrt(2) * sigma due to the sum of variances
                    prob = norm.cdf(mu_diff / sigma_diff)
                    return prob
                
                sigma = 0.05
                
                n = logits.shape[0]
                pairs = list(itertools.combinations(range(n), 2))

                losses = []
                for i, j in pairs:
                    # Ranking loss
                    # Compute the difference in logits for each pair
                    output_i = logits[i].to(labels.device)
                    output_j = logits[j].to(labels.device)

                    label_i = labels[i].cpu()
                    label_j = labels[j].cpu()
                    
                    # Calculate the probability that label_i > label_j
                    prob = calculate_probability(label_i, label_j, sigma)
                    target = torch.tensor([prob]).to(output_i.device)
                    # Ranking loss between target and output logits
                    ranking_loss = BCEWithLogitsLoss()(output_i - output_j, target)
                    
                    # Contrastive loss
                    # Get embeddings
                    hidden_state_i =  torch.mean(hidden_states[i],dim=0)
                    hidden_state_j =  torch.mean(hidden_states[j],dim=0)                  
                    
                    # Calculate cosine similarity between embeddings: similar 1 / dissimilar value -1
                    cos_sim = F.cosine_similarity(hidden_state_i, hidden_state_j,dim=0)
                    # transform target to a binary: similar 0 / dissimilar value 1
                    target = ((target - 0.5) * 2) ** 2
                    
                    #calculate loss
                    margin = 0.5
                    contrastive_loss = target * ((F.relu(cos_sim - margin))**2)

                    gamma = 10  # contrastive_loss weighting
                    loss = ranking_loss + contrastive_loss * gamma
                    losses.append(loss)
                   

                average_loss = torch.stack(losses).mean()
                loss = average_loss

        if not return_dict:
            output = (logits,) + outputs[1:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states, #outputs.[0]
            attentions=outputs.attentions,
        )

### Load T5 model
this creates a T5 model with prediction head and LoRA modification

In [6]:
def load_T5_model(checkpoint, num_labels, full=False):
    
    # Load model and tokenizer
    model = T5EncoderModel.from_pretrained(checkpoint)
    tokenizer = T5Tokenizer.from_pretrained(checkpoint)                

    
    # Create new Classifier model with PT5 dimensions
    class_config=ClassConfig(num_labels=num_labels)
    class_model=T5EncoderForSimpleSequenceToRank(model.config,class_config)
    
    # Set encoder and embedding weights to checkpoint weights
    class_model.shared=model.shared
    class_model.encoder=model.encoder    
    
    # Delete the checkpoint model
    model = class_model
    del class_model
    
    if full == True:
        return model, tokenizer 
    
    # Print number of trainable parameters
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    print("T5_Classfier\nTrainable Parameter: "+ str(params))    
 
    # lora modification
    peft_config = LoraConfig(
        r=4, lora_alpha=1, bias="all", target_modules=["q","k","v","o"]
    )
    
    # model = inject_adapter_in_model(peft_config, model)
    model = PeftModelForSequenceClassification(model, peft_config)

    # Print trainable Parameter          
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    print("T5_LoRA_Classfier\nTrainable Parameter: "+ str(params) + "\n")
    
    return model, tokenizer

# Training Definition 

## Training functions

In [7]:
def load_model(checkpoint, filepath):
# Creates a new PT5 model and loads the finetuned weights from a file

    model = T5EncoderModel.from_pretrained(checkpoint)
    tokenizer = T5Tokenizer.from_pretrained(checkpoint)

    # Create new Classifier model with PT5 dimensions
    class_config=ClassConfig(num_labels=1)
    class_model=T5EncoderForSimpleSequenceToRank(model.config,class_config)

    # Set encoder and embedding weights to checkpoint weights
    class_model.shared=model.shared
    class_model.encoder=model.encoder    

    # Delete the checkpoint model
    model = class_model
    del class_model

    peft_model = PeftModelForSequenceClassification.from_pretrained(model, filepath)
    del model

    return tokenizer, peft_model


def find_checkpoint_folder(base_path):
    # Iterate over all directories in the base path
    for folder_name in os.listdir(base_path):
        # Check if the folder name starts with "checkpoint"
        if folder_name.startswith("checkpoint"):
            return folder_name
    return None

In [8]:
# Set random seeds for reproducibility of your trainings run
def set_seeds(s):
    torch.manual_seed(s)
    np.random.seed(s)
    random.seed(s)
    set_seed(s)

# Dataset creation
def create_dataset(tokenizer,seqs,labels):
    tokenized = tokenizer(seqs, max_length=1024, padding=True, truncation=True)
    dataset = Dataset.from_dict(tokenized)
    dataset = dataset.add_column("labels", labels)

    return dataset

   
# Main training fuction
def train_per_protein(
        checkpoint,       #model checkpoint
        data_path,        #dataset name
        save_path,        #path to save your model
    
        num_labels = 1,   #1 for regression, >1 for classification
    
        # effective training batch size is batch * accum
        # we recommend an effective batch size of 8 
        batch = 4,        #for training
        accum = 2,        #gradient accumulation
    
        val_batch = 16,   #batch size for evaluation
        epochs = 10,      #training epochs
        lr = 3e-4,        #recommended learning rate
        seed = 42,        #random seed
        mixed = True,     #enable mixed precision training
        full = False,     #enable training of the full model (instead of LoRA)
        gpu = 1 ):        #gpu selection (1 for first gpu)

    print("Model used:", checkpoint, "\n")

    # Set gpu device
    os.environ["CUDA_VISIBLE_DEVICES"]=str(gpu-1)
    
    # Set all random seeds
    set_seeds(seed)
    
    # load model
    model, tokenizer = load_T5_model(checkpoint, num_labels, full)
        
    # Load data
    train_df, valid_df = read_data(path)
    
    if (len(train_df) % batch) == 1:
        train_df = train_df.iloc[:-1]
        print("dropped last training example")
    
    # Preprocess inputs
    # Replace uncommon AAs with "X"
    train_df["sequence"]=train_df["sequence"].str.replace('|'.join(["O","B","U","Z","J"]),"X",regex=True)
    valid_df["sequence"]=valid_df["sequence"].str.replace('|'.join(["O","B","U","Z","J"]),"X",regex=True)

    # Add whitespaces between AAs 
    train_df['sequence']=train_df.apply(lambda row : " ".join(row["sequence"]), axis = 1)
    valid_df['sequence']=valid_df.apply(lambda row : " ".join(row["sequence"]), axis = 1)

    # Create Datasets
    train_set=create_dataset(tokenizer,list(train_df['sequence']),list(train_df['label']))
    valid_set=create_dataset(tokenizer,list(valid_df['sequence']),list(valid_df['label']))
    
    # Specify the path where you want to create the folder
    path_to_create = save_path + "/" + data_path + "/" + str(seed)
    
    # Check if the directory already exists
    if not os.path.exists(path_to_create):
        # Create the directory
        os.makedirs(path_to_create)
        print(f"Directory '{path_to_create}' created.")
    else:
        print(f"Directory '{path_to_create}' already exists.")

    # Huggingface Trainer arguments
    args = TrainingArguments(
        path_to_create,
        eval_strategy = "steps",
        eval_steps = 3000 / batch / accum,
        logging_strategy = "epoch",
        save_strategy = "steps",
        save_steps  = 3000 / batch / accum,
        learning_rate = lr,
        lr_scheduler_type = "cosine",
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=val_batch,
        gradient_accumulation_steps=accum,
        num_train_epochs=epochs,
        seed = seed,
        fp16 = mixed,
        metric_for_best_model="spearmanr",
        load_best_model_at_end=True,
        save_total_limit=1,
    ) 

    # Metric definition for validation data
    def compute_metrics(eval_pred):
        
        metric = load("spearmanr")
        predictions, labels = eval_pred

        return metric.compute(predictions=predictions, references=labels)
    
    early_stop = EarlyStoppingCallback(10, 0)
    
    # Trainer          
    trainer = Trainer(
        model,
        args,
        train_dataset=train_set,
        eval_dataset=valid_set,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[early_stop]
    )    
    
    # Train model
    trainer.train()


    return tokenizer, model, trainer.state.log_history


# Run Training

## Training

In [9]:
os.listdir("./data/raw/")

['CAPSD_AAV2S_Sinai_2021',
 'GFP_AEQVI_Sarkisyan_2016',
 'HIS7_YEAST_Pokusaeva_2019',
 'PHOT_CHLRE_Chen_2023']

In [10]:
path = os.listdir("./data/raw/")[3]
path

'PHOT_CHLRE_Chen_2023'

In [11]:
save_path = "./models/ProtT5_finetuning/checkpoints/PT5-LoRA_Rank_contrastive_onesided"
save_path

'./models/ProtT5_finetuning/checkpoints/PT5-LoRA_Rank_contrastive_onesided'

In [12]:
random.seed(42)
seeds = [random.randint(0, 1000) for _ in range(3)]
seeds

[654, 114, 25]

In [13]:
GPU = 1

In [ ]:
# run training for all seeds
for s in seeds:
        print("SEED:",s)
        print("Dataset:", path)
        print("GPU:", GPU)
    
        tokenizer, model, history = train_per_protein(checkpoint, path, save_path, batch = 4, accum = 2, epochs = 100, seed = s, mixed = True, gpu=GPU)

        
        # Specify the path where you want to create the folder
        path_to_create = save_path + "/history"

        # Check if the directory already exists
        if not os.path.exists(path_to_create):
            # Create the directory
            os.makedirs(path_to_create)
            print(f"Directory '{path_to_create}' created.")
        else:
            print(f"Directory '{path_to_create}' already exists.")

        # Save the training history to a file
        with open(save_path + "/history/" + path + "_" +  str(s) + "_LoRA.csv", "w") as f:
            json.dump(history, f)
        
        del tokenizer, model, history

    

# Plots

In [14]:
def plot_res(save_path, path):# Get loss, val_loss, and the computed metric from history
    
    with open(save_path + "/history/"+ path, "r") as f:
        history = json.load(f)
    
    loss = [x['loss'] for x in history if 'loss' in x]
    val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

    # Get spearman (for ranking) 
    metric = [x['eval_spearmanr'] for x in history if 'eval_spearmanr' in x]

    epochs = [x['epoch'] for x in history if 'loss' in x]
    epochsv = [x['epoch'] for x in history if 'eval_loss' in x]

    # Create a figure with two y-axes
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    # Plot loss and val_loss on the first y-axis
    line1 = ax1.plot(epochs, loss, label='train_loss')
    line2 = ax1.plot(epochsv, val_loss, label='val_loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')

    # Plot the computed metric on the second y-axis
    line3 = ax2.plot(epochsv, metric, color='red', label='val_spearman')
    ax2.set_ylabel('Spearman')
    ax2.set_ylim([0, 1])

    # Combine the lines from both y-axes and create a single legend
    lines = line1 + line2 + line3
    labels = [line.get_label() for line in lines]
    ax1.legend(lines, labels, loc='lower left')

    # Show the plot
    plt.title("Training History - "+ path)
    plt.show()

In [ ]:
files = os.listdir(save_path + "/history/")
files = [entry for entry in files if path in entry]

for f in files:
    plot_res(save_path,f)

# Compute Validation/Test predictions

In [15]:
# set GPU
GPU = 1
os.environ["CUDA_VISIBLE_DEVICES"]=str(GPU-1)

In [16]:
os.listdir("./data/raw/")

['CAPSD_AAV2S_Sinai_2021',
 'GFP_AEQVI_Sarkisyan_2016',
 'HIS7_YEAST_Pokusaeva_2019',
 'PHOT_CHLRE_Chen_2023']

In [17]:
# set dataset
path = os.listdir("./data/raw/")[3]
path

'PHOT_CHLRE_Chen_2023'

In [18]:
save_path

'./models/ProtT5_finetuning/checkpoints/PT5-LoRA_Rank_contrastive_onesided'

In [19]:
# load valid and test data
def read_prediction_data(path):
    
    df_valid = pd.read_csv("./data/splits/" + path + "/valid.csv")
    df_test = pd.read_csv("./data/splits/" + path + "/test.csv")

    df_valid = df_valid.rename(columns={"mutated_sequence" : "sequence", "DMS_score" : "label"})    
    df_test = df_test.rename(columns={"mutated_sequence" : "sequence", "DMS_score" : "label"})
    
    valid_df["set"] = "valid"
    test_df["set"] = "test"
    
    combined = pd.concat([valid_df[["mutant","set","sequence","label"]], test_df[["mutant","set","sequence","label"]]])

    return combined

def compute_prediction(path, model, tokenizer):
    
    combined = read_prediction_data(path)
    
    # Preprocess sequences
    combined.loc[:,"sequence"]=combined["sequence"].str.replace('|'.join(["O","B","U","Z","J"]),"X",regex=True)
    combined.loc[:,'sequence']=combined.apply(lambda row : " ".join(row["sequence"]), axis = 1)
    
    # Set the device to use
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    
    model.to(device)

    # create Dataset
    combined_set=create_dataset(tokenizer,list(combined['sequence']),list(combined['label']))
    # make compatible with torch DataLoader
    combined_set = combined_set.with_format("torch", device=device)

    # Create a dataloader for the test dataset
    dataloader = DataLoader(combined_set, batch_size=8, shuffle=False)

    # Put the model in evaluation mode
    model.eval()

    # Make predictions on the test dataset
    predictions = []
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            #add batch results(logits) to predictions
            predictions += model.float()(input_ids, attention_mask=attention_mask).logits.tolist()
            
    return predictions      

In [ ]:
# compute predictions for all seeds
combined_df = read_prediction_data(path)

for s in seeds:
    path_cp_lora = save_path + "/" + path +"/" + str(s)
    print("Seed:", s)
    
    tokenizer, model = load_model(checkpoint, path_cp_lora + "/" + find_checkpoint_folder(path_cp_lora))
    
    pred = compute_prediction(path, model, tokenizer)
    
    model.to("cpu")
    del model, tokenizer
    
    print(stats.spearmanr(a=pred, b=combined_df.label, axis=0))    
    
    combined_df["seed_"+ str(s)] =  [item for sublist in pred for item in sublist]


In [ ]:
# to save memory for the results we dropped the seqeunce and label columns
# with the mutant column available each row can be linked back to the data/splits files
combined_df.drop(["sequence","label"], axis=1, inplace=True)
combined_df